### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

c:\D_Drive\RAGUdemy\Agentic_AI\agenticai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds, right? Some species like budgies, macaws, and cockatiels are known for their ability to mimic human speech. But why do they do that?\n\nFirst, I might think about their natural behavior. In the wild, parrots are social animals. They live in flocks, so communication is essential for them. Maybe talking is a form of social bonding? Like how they use calls to stay in touch with their flock. But how does that translate to mimicking human speech when kept as pets?\n\nI remember reading that parrots have a part of their brain called the "sound box" or syrinx, which helps them produce sounds. They can learn various sounds, not just human speech. So perhaps their ability to mimic is a learned behavior. But why specifically human speech? Is it just because they can, or is there a purpose?\n\nAlso, some sources mention that in the wild, parrots might mi

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'sgqtvhsfa', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 153, 'total_tokens': 239, 'completion_time': 0.136227378, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006465858, 'prompt_tokens_details': None, 'queue_time': 0.162224662, 'total_time': 0.142693236}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d720e-cb8b-70f0-b

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny today! You won't need an umbrella, but you might want to apply sunscreen if heading outdoors.
